# Sanhedrin LLM - Vollständige Deep Learning Klassifikation hebräischer Fragmente

**Kompletter, lauffähiger Code für github.com/Rosary-mom/RosaryxAI**

Dieses Notebook enthält das vollständige Training mit realen Geniza-Daten, Transfer-Learning und Inference für hochgeladene Fragmente.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from PIL import Image
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

classes = [
    'Oriental_Square', 'Oriental_Semi', 'Oriental_Cursive',
    'Ashkenazic_Square', 'Ashkenazic_Semi', 'Ashkenazic_Cursive',
    'Sephardic_Square', 'Sephardic_Semi', 'Sephardic_Cursive',
    'Italian_Square', 'Italian_Semi', 'Italian_Cursive',
    'Byzantine_Semi', 'Yemenite_Square'
]

In [ ]:
data_dir = 'geniza_patches'  # Pfad zu echten Fragmenten

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(data_dir, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50):
    best_acc = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        acc = 100 * correct / total
        print(f'Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}%')
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), 'sanhedrin_hebrew_classifier.pth')
    return model

model = train_model(model, train_loader, val_loader, epochs=50)

In [ ]:
def classify_uploaded_fragment(image_path):
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model.load_state_dict(torch.load('sanhedrin_hebrew_classifier.pth', map_location=device))
    model.eval()
    transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    img = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img)
        pred = torch.argmax(output, 1).item()
    return classes[pred]

# Beispiel: classify_uploaded_fragment('fragment.jpg')